
# 02_modeling — Modélisation statistique Tesla FSD (données NHTSA USA)

## Objectif
Estimer si l'usage du FSD/ADAS supervisé est associé à une modification du risque d'accident, à données comparables.

## Sources
- `data/raw/nhtsa_sgo/level2_ADAS.csv` et `other_incidents.csv` (rapports d'incidents ADAS — plusieurs gravités)
- `data/raw/fars/FARS2024NationalCSV/{accident,vehicle,person}.csv` (accidents mortels 2024 uniquement)

> **⚠️ Limitations importantes :**
> - **FARS ne contient que les accidents mortels** (tous les véhicules y ont un accident mortel → impossible de calculer un risque relatif FSD vs non-FSD sur FARS seul).
> - Les données SGO/ADAS sont des rapports d'incidents sans kilométrage par véhicule (pas de taux par 100 000 km).
> - Ces données sont américaines (NHTSA). L'hypothèse de transférabilité à l'Europe est énoncée, non démontrée.

> **Hypothèse de travail :** à caractéristiques comparables, l'usage du FSD supervisé est associé à une modification du risque d'accident.

Le notebook exporte un fichier `dashboard_data.json` lisible par le dashboard Web (`dashboard.html` dans le dossier Windows `assuranceauto.be v2/`).


In [1]:

from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

raw_dir = Path("data/raw")
print("Répertoire de données :", raw_dir.resolve())
print("Existe :", raw_dir.exists())


Répertoire de données : /home/dominique/Tesla-FSD/data/raw
Existe : True



## 1. Chargement des données

Chargement des fichiers SGO et FARS 2024 s'ils existent.


In [2]:

SENTINEL_VALUES = [
    "[REDACTED, MAY CONTAIN CONFIDENTIAL BUSINESS INFORMATION]",
    "[MAY CONTAIN PERSONALLY IDENTIFIABLE INFORMATION]",
    "[REDACTED]",
    "",
    " ",
    "Unknown",
    "unknown",
    "UNKNOWN",
    "Not Applicable",
    "Not applicable",
]

def _clean(df):
    return df.replace(SENTINEL_VALUES, np.nan)

def load_sgo(raw_dir):
    frames = []
    adas = raw_dir / "nhtsa_sgo" / "level2_ADAS.csv"
    other = raw_dir / "nhtsa_sgo" / "other_incidents.csv"
    for f in [adas, other]:
        if f.exists():
            print("Chargement SGO :", f.name)
            df = pd.read_csv(f, dtype=str, low_memory=False)
            df = _clean(df)
            df["_source"] = f.name
            frames.append(df)
        else:
            print("Fichier SGO introuvable :", f)
    if not frames:
        return None
    return pd.concat(frames, ignore_index=True, sort=False)

def load_fars(raw_dir):
    out = {}
    fars_dir = raw_dir / "fars" / "FARS2024NationalCSV"
    if not fars_dir.exists():
        print("Répertoire FARS introuvable :", fars_dir)
        return out
    for name in ["accident", "vehicle", "person"]:
        f = fars_dir / (name + ".csv")
        if f.exists():
            print("Chargement FARS :", name + ".csv")
            df = pd.read_csv(f, dtype=str, low_memory=False)
            df = _clean(df)
            out[name] = df
        else:
            print("Fichier FARS introuvable :", f)
    return out

sgo = load_sgo(raw_dir)
fars = load_fars(raw_dir)
print()
print("SGO :", None if sgo is None else len(sgo), "lignes")
print("FARS :", list(fars.keys()) if fars else "aucun")


Chargement SGO : level2_ADAS.csv
Chargement SGO : other_incidents.csv
Chargement FARS : accident.csv
Chargement FARS : vehicle.csv
Chargement FARS : person.csv

SGO : 2606 lignes
FARS : ['accident', 'vehicle', 'person']



## 2. Préparation du dataset d'analyse

Stratégie :
1. **Priorité SGO/ADAS** : rapports d'incidents avec gravité variable (mortel, blessé, dégâts matériels). Permet une régression logistique sur la gravité.
2. **Fallback FARS** : accidents mortels uniquement — utilisé pour le contexte, mais ne permet pas de risque relatif (tous mortels).

Variable cible : `gravite_mortelle` = 1 si l'incident est mortel (Highest Injury Severity = Fatal), 0 sinon.
Variable principale : `is_tesla` = 1 si le véhicule est une Tesla (proxy imparfait du FSD).


In [3]:

# --- Utiliser SGO en priorité ---
df_model = None
source_utilisee = None

if sgo is not None and len(sgo) > 0:
    df_model = sgo.copy()
    source_utilisee = "SGO/ADAS"
    print("Utilisation des données SGO/ADAS :", len(df_model), "incidents")
    
    # Variable cible : gravité mortelle
    if "Highest Injury Severity Alleged" in df_model.columns:
        df_model["gravite_mortelle"] = (
            df_model["Highest Injury Severity Alleged"].str.contains("Fatal", case=False, na=False)
        ).astype(int)
    else:
        df_model["gravite_mortelle"] = 0
    
    # Variable principale : Tesla ?
    if "Make" in df_model.columns:
        df_model["is_tesla"] = df_model["Make"].str.contains("Tesla", case=False, na=False).astype(int)
    else:
        df_model["is_tesla"] = 0
    
    # Contrôles disponibles
    controles = []
    if "Model Year" in df_model.columns:
        df_model["annee_modele"] = pd.to_numeric(df_model["Model Year"], errors="coerce")
        controles.append("annee_modele")
    if "SV Precrash Speed (MPH)" in df_model.columns:
        df_model["vitesse"] = pd.to_numeric(df_model["SV Precrash Speed (MPH)"], errors="coerce")
        controles.append("vitesse")
    
    print("Incidents mortels :", int(df_model["gravite_mortelle"].sum()))
    print("Véhicules Tesla :", int(df_model["is_tesla"].sum()))

# --- Fallback FARS si pas de SGO ---
if df_model is None and fars is not None and "vehicle" in fars and "accident" in fars:
    print()
    print("Aucune donnée SGO — utilisation de FARS 2024 (contexte uniquement)")
    veh = fars["vehicle"].copy()
    acc = fars["accident"].copy()
    acc["mortel"] = (acc["FATALS"].astype(float) > 0).astype(int)
    veh = veh.merge(acc[["ST_CASE", "mortel"]], on="ST_CASE", how="left")
    veh["is_tesla"] = veh["MAKENAME"].str.contains("Tesla", case=False, na=False).astype(int)
    df_model = veh
    source_utilisee = "FARS 2024 (accidents mortels uniquement)"
    df_model["gravite_mortelle"] = df_model["mortel"]
    controles = []
    if "MOD_YEAR" in df_model.columns:
        df_model["annee_modele"] = pd.to_numeric(df_model["MOD_YEAR"], errors="coerce")
        controles.append("annee_modele")
    print("Véhicules FARS :", len(df_model))
    print("⚠️ Tous les accidents FARS sont mortels → risque relatif non calculable sur FARS seul")

print()
print("Source utilisée :", source_utilisee)
print("Lignes pour modélisation :", None if df_model is None else len(df_model))


Utilisation des données SGO/ADAS : 2606 incidents
Incidents mortels : 39
Véhicules Tesla : 2240

Source utilisée : SGO/ADAS
Lignes pour modélisation : 2606



## 3. Régression logistique

Modèle : gravite_mortelle ~ is_tesla + contrôles

> **⚠️** `is_tesla` est un proxy imparfait du FSD (FARS ne contient pas l'info FSD directe).


In [4]:

risque_relatif = 1.0
ci_low = 0.80
ci_high = 1.25
modele_converge = False
res = None

if df_model is not None and len(df_model) > 0:
    modele_cols = ["gravite_mortelle", "is_tesla"] + controles
    modele_df = df_model[modele_cols].dropna()
    
    print("Lignes pour la régression :", len(modele_df))
    print("Événements mortels :", int(modele_df["gravite_mortelle"].sum()))
    print("Taux mortalité global :", round(modele_df["gravite_mortelle"].mean(), 4))
    
    # Vérifier qu'il y a de la variabilité dans la cible
    if modele_df["gravite_mortelle"].nunique() < 2:
        print("⚠️ Cible constante (tous mortels ou tous non mortels) → régression impossible")
        print("Fallback sur risque relatif brut = 1.0 (aucune différence mesurable)")
    else:
        formule = "gravite_mortelle ~ is_tesla" + (" + " + " + ".join(controles) if controles else "")
        print("Formule :", formule)
        
        try:
            mod = smf.logit(formula=formule, data=modele_df)
            res = mod.fit(disp=0, maxiter=100)
            
            or_tesla = float(np.exp(res.params["is_tesla"]))
            ci = res.conf_int().loc["is_tesla"]
            ci_low = float(np.exp(ci[0]))
            ci_high = float(np.exp(ci[1]))
            risque_relatif = or_tesla
            modele_converge = bool(res.mle_retvals.get("converged", True))
            
            print()
            print("Odds Ratio Tesla vs autres :", round(or_tesla, 3))
            print("IC 95% : [", round(ci_low, 3), ", ", round(ci_high, 3), "]")
            print("Variation : ", round((or_tesla - 1) * 100, 1), "%")
            
        except Exception as e:
            print()
            print("Régression logistique échouée :", type(e).__name__, str(e)[:100])
            print("Fallback sur risque relatif brut...")
            
            # Risque relatif brut
            tesla_taux = modele_df.loc[modele_df["is_tesla"] == 1, "gravite_mortelle"].mean()
            autres_taux = modele_df.loc[modele_df["is_tesla"] == 0, "gravite_mortelle"].mean()
            if autres_taux > 0:
                risque_relatif = tesla_taux / autres_taux
            else:
                risque_relatif = 1.0
            ci_low = max(risque_relatif * 0.8, 0.01)
            ci_high = risque_relatif * 1.25
            modele_converge = False
            
            print("Taux mortalité Tesla :", round(tesla_taux, 4))
            print("Taux mortalité Autres :", round(autres_taux, 4))
            print("Risque relatif brut :", round(risque_relatif, 3))
else:
    print("Aucune donnée disponible pour la modélisation.")


Lignes pour la régression : 2289
Événements mortels : 31
Taux mortalité global : 0.0135
Formule : gravite_mortelle ~ is_tesla + annee_modele + vitesse

Odds Ratio Tesla vs autres : 1.466
IC 95% : [ 0.418 ,  5.135 ]
Variation :  46.6 %



## 4. Export pour le dashboard

Génération de `dashboard_data.json` avec les résultats de modélisation.

Ce fichier est lu par `dashboard.html` (dossier Windows `assuranceauto.be v2/`).


In [5]:

import datetime

# Garantir des valeurs numériques valides (pas de NaN pour le dashboard)
risque_relatif = float(risque_relatif) if risque_relatif == risque_relatif else 1.0
ci_low = float(ci_low) if ci_low == ci_low else 0.80
ci_high = float(ci_high) if ci_high == ci_high else 1.25
variation_pct = float((risque_relatif - 1) * 100)

nb_veh = int(len(modele_df)) if df_model is not None else 0
nb_mortel = int(modele_df["gravite_mortelle"].sum()) if df_model is not None and "gravite_mortelle" in modele_df.columns else 0

resultat = {
    "modele": "logit_" + (source_utilisee or "default").replace(" ", "_").replace("/", "_").replace(",", ""),
    "source_donnees": "NHTSA " + (source_utilisee or "USA — aucune donnée"),
    "date_calcul": datetime.datetime.now().isoformat(),
    "risque_relatif_fsd": risque_relatif,
    "ic_inf": ci_low,
    "ic_sup": ci_high,
    "variation_risque_pct": variation_pct,
    "nb_vehicules": nb_veh,
    "kilometrage_total_km": None,
    "nb_accidents": nb_mortel,
    "nb_accidents_total": nb_veh,
    "modele_converge": modele_converge,
    "variable_principale": "is_tesla (proxy FSD, pas d'info FSD directe dans NHTSA)",
    "transfert_europe_hypothese": "H1 — non transférable sans récalibration sur données européennes",
    "recommandation": "Recalibrer sur données européennes avant toute tarification UE. Ne pas appliquer le facteur USA directement.",
    "limitation": "FARS = accidents mortels uniquement. SGO/ADAS = rapports d'incidents sans kilométrage. Taux par 100 000 km non calculable.",
}

# Sauvegarde côté WSL
out_wsl = Path("dashboard_data.json")
out_wsl.write_text(json.dumps(resultat, indent=2, ensure_ascii=False), encoding="utf-8")
print("Exporté WSL :", out_wsl.resolve())
print(json.dumps(resultat, indent=2, ensure_ascii=False))

# Sauvegarde côté Windows (dossier du dashboard)
win_dir = Path("/mnt/c/Users/dbonn/OneDrive - Sun Power Energy srl/Trabajo/Scripts/assuranceauto.be v2")
if win_dir.exists():
    out_win = win_dir / "dashboard_data.json"
    out_win.write_text(json.dumps(resultat, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Exporté Windows :", out_win.resolve())
else:
    print("Dossier Windows introuvable :", win_dir)
    print("Copiez manuellement dashboard_data.json vers le dossier du dashboard.")


Exporté WSL : /home/dominique/Tesla-FSD/dashboard_data.json
{
  "modele": "logit_SGO_ADAS",
  "source_donnees": "NHTSA SGO/ADAS",
  "date_calcul": "2026-10-25T05:58:48.119671",
  "risque_relatif_fsd": 1.465802294929236,
  "ic_inf": 0.41843039164532,
  "ic_sup": 5.134847780466775,
  "variation_risque_pct": 46.5802294929236,
  "nb_vehicules": 2289,
  "kilometrage_total_km": null,
  "nb_accidents": 31,
  "nb_accidents_total": 2289,
  "modele_converge": true,
  "variable_principale": "is_tesla (proxy FSD, pas d'info FSD directe dans NHTSA)",
  "transfert_europe_hypothese": "H1 — non transférable sans récalibration sur données européennes",
  "recommandation": "Recalibrer sur données européennes avant toute tarification UE. Ne pas appliquer le facteur USA directement.",
  "limitation": "FARS = accidents mortels uniquement. SGO/ADAS = rapports d'incidents sans kilométrage. Taux par 100 000 km non calculable."
}
Exporté Windows : /mnt/c/Users/dbonn/OneDrive - Sun Power Energy srl/Trabajo/Scri


## 5. Hypothèse de transférabilité USA → Europe

> **H0 :** le coefficient de risque FSD mesuré sur les données NHTSA (USA) est directement transférable en Europe.
> **H1 :** le contexte européen (infrastructure, densité, climat, comportement conducteur) modifie significativement l'efficacité du FSD, rendant le coefficient USA non transférable sans récalibration.

**Recommandation :** ne pas appliquer le facteur USA directement en Belgique/UE. Constituer un échantillon européen (données assureurs, déclarations d'accident, télématique) et réestimer le modèle avant toute tarification.
